# Code was run on Colab Pro

In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import collections
import torch.optim as optim
from torch.optim import Optimizer
import time
import matplotlib.pyplot as plt

from AdamW          import AdamW
from utils          import utility, misreportUtility, misreportOptimization, trueUtility, loss
from networks       import AdditiveMechanism, Misreports,AllocationNet,PaymentNet
from restrictedAdam import Adam 
from networks import MixedWrapper

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.cuda.set_device(1)

# Set Random Seed 

In [3]:
# Initializing seeds
torch.manual_seed(4)
np.random.seed(4)

# Testing Function

In [4]:
def test(nBatch, nbrInit, R, gamma=0.001, minimum=0, maximum=1):
    """
    Evaluate both the pure neural network (Net) and the mixed mechanism (Mixed).
    - Keep the original misreportUtility / utility / loss implementation.
    - Disable straight-through for mixed evaluation (hard selection).
    """

    # Comment removed.

    reserve=0.5
    true = np.random.rand(nBatch, nAgent, nObject)
    localMisreports     = np.random.rand(nBatch, nbrInit, nAgent, nObject)
    batchMisreports     = torch.tensor(localMisreports).float().to(device)
    batchTrueValuations = torch.tensor(true).float().to(device)
    batchMisreports.requires_grad = True

    def _eval_once(mech_callable):
        # Misreport optimization followed by regret/payment/loss evaluation.
        opt = Adam([batchMisreports], lr=gamma)
        for k in range(R):
            advU = misreportUtility(mech_callable, batchTrueValuations, batchMisreports)
            los  = -1*torch.mean(advU).to(device)
            los.backward()
            opt.step(restricted=True, min=minimum, max=maximum)
            opt.zero_grad()

        misReportUtilityMax  = torch.max(advU, dim=1)[0]
        allocation, payment = mech_callable(batchTrueValuations)
        regret = F.relu(misReportUtilityMax - utility(batchTrueValuations, allocation, payment))
        mregret = torch.sum(torch.mean(regret, dim=0)).to(device)
        mregret = float(mregret.detach().cpu().numpy())
        with torch.no_grad():
            l, rMean, p = loss(payment, regret)
        return mregret, float(p.detach().cpu().numpy()), float((-l).detach().cpu().numpy())**2

    # ----- Define two mechanism callables -----
    # Pure neural network
    if hasattr(mechanism, "base"):   # Comment removed.

        def mech_net(X):  # Pure neural network
            return mechanism.base(X)
    else:
        def mech_net(X):  # Use the mechanism itself if it is not wrapped.
            return mechanism(X)

    # Mixed mechanism with hard profile-wise selection.
    st_backup = getattr(mechanism, "st", None)
    if hasattr(mechanism, "st"):
        mechanism.st = False
    def mech_mix(X):
        return mechanism(X)
    # --------------------

    # ----- Evaluate both mechanisms -----
    net_reg, net_pay, net_optrev   = _eval_once(mech_net)
    mix_reg, mix_pay, mix_optrev   = _eval_once(mech_mix)

    # Restore straight-through state if needed.
    if hasattr(mechanism, "st"):
        mechanism.st = st_backup if st_backup is not None else True

    # Print both results.
    print(f"[NET  ] regret={net_reg:.5f}  avg/bidder={net_reg/nAgent:.5f}  optRev={net_optrev:.3f}  payment={net_pay:.3f}")
    print(f"[MIXED] regret={mix_reg:.5f}  avg/bidder={mix_reg/nAgent:.5f}  optRev={mix_optrev:.3f}  payment={mix_pay:.3f}")
    testRegret.append(net_reg)
    testPayment.append(net_pay)
    testOptimal.append(net_optrev)

    # Return a dictionary for optional logging.
    return {
        "net":   {"regret": net_reg, "payment": net_pay, "optrev": net_optrev},
        "mixed": {"regret": mix_reg, "payment": mix_pay, "optrev": mix_optrev},
    }

def finaltest(nBatch, nbrInit, R, gamma=0.001, minimum=0, maximum=1):
    
    """ This function computes the regret and payment of mechanism on a test set of size nBatch
        The optimal misreport is computed by optimizing the utility function (not by using the Misreport network)
        for R gradient steps (of stepsize gamma) and starting from nbrInit initialization, we only keep the best misreport
        To compute the regret we evaluate the mechanism at the misreport and compare to the valuation
        minimum and maximum indicate the range of the valuations
    """
    
    true = np.random.rand(nBatch,nAgent,nObject)

    # Final evaluation uses the pure neural student, not the Stage III mixed wrapper.
    eval_mechanism = mechanism.base if hasattr(mechanism, "base") else mechanism
    eval_mechanism.eval()

   
    with torch.no_grad():
        batchTrue = torch.tensor(true).float().to(device)
        allocation, payment = eval_mechanism(batchTrue)
        batchTrueValuations = torch.tensor(true).float().to(device).unsqueeze(1).repeat(1,1000, 1, 1)
        localMisreports     =  np.expand_dims(true,1).repeat(1000,axis=1)
        max_u = torch.tensor(np.zeros((nBatch,nAgent,nObject))).float().to(device)
        best_misreport_values = torch.zeros(nBatch, nAgent, nObject).to(device)
        for l in range(nAgent):
            for i in range(nObject):
                localMisreports     = np.expand_dims(true,1).repeat(1000,axis=1)
                localMisreports[:,:,l,i]=0
                for k in range(nBatch):
                    for j in range(999):
                        localMisreports[k,j+1,l,i]=  localMisreports[k,j,l,i]+0.001

                batchMisreports     = torch.tensor(localMisreports).float().to(device)
            
                a_m,p_m = eval_mechanism(batchMisreports.reshape(-1,nAgent,nObject))
                utility1 = (a_m.reshape(nBatch,1000,nAgent,nObject)*batchTrueValuations).sum(dim=3) - p_m.reshape(nBatch,1000,nAgent)
                
                
                cur_u = torch.max(utility1,dim=1)[0][:,l]-utility(batchTrue, allocation, payment)[:,l]
                max_u[:,l,i] = cur_u
                best_idx = torch.argmax(utility1[:,:,l], dim=1)

                # Extract the selected misreport value.
                best_values = batchMisreports[
                    torch.arange(nBatch).to(device),
                    best_idx,
                            l,
                    i
                ]

                best_misreport_values[:,l,i] = best_values
        
        regret = F.relu(torch.sum(torch.max(max_u,axis=2)[0])/nBatch)
        mregret= float(regret.cpu().detach().numpy())

        regret = F.relu(torch.sum(torch.max(max_u,axis=2)[0])/nBatch)
        mregret= float(regret.cpu().detach().numpy())
        with torch.no_grad():
            l,rMean,p = loss(payment, regret)


    batchTrueValuations = torch.tensor(true, dtype=torch.float32, device=device)  # [B,A,M]
    full_best = best_misreport_values                                            # [B,A,M]

    B = nBatch
    A = nAgent
    M = nObject
    m = nObject

    mis_list = []

    # Candidate 1: use the best value for all items.
    mis_full = full_best                                     # [B,A,M]
    mis_list.append(mis_full)
    # Candidate 2: change one item at a time and keep the others truthful.
    for i_obj in range(m):
        mis_i = batchTrueValuations.clone()            # [B, nAgent, nObject]
        mis_i[:, :, i_obj] = full_best[:, :, i_obj]    # Change only item i_obj.
        mis_list.append(mis_i)

    # Candidate 3: for each bidder, change only the item with the largest regret.
    best_item_idx = torch.argmax(max_u, dim=2)               # [B,A]
    mis_bestitem = batchTrueValuations.clone()               # [B,A,M]
    b_idx = torch.arange(B, device=device)
    for l in range(A):
        idx_l = best_item_idx[:, l]                          # [B]
        mis_bestitem[b_idx, l, idx_l] = full_best[b_idx, l, idx_l]
    mis_list.append(mis_bestitem)

# Optional global random misreport initializations.
    num_global_random = 1
    for _ in range(num_global_random):
        mis_rand = torch.rand(B, A, M, device=device)
        mis_list.append(mis_rand)

# Optional local perturbations around truthful bids.
    num_local_around_truth = 1
    noise_scale = 0.2
    for _ in range(num_local_around_truth):
        noise = torch.randn(B, A, M, device=device) * noise_scale
        mis_loc = batchTrueValuations + noise
        mis_loc = torch.clamp(mis_loc, 0.0, 1.0)
        mis_list.append(mis_loc)

# Optional local perturbations around the best misreports.
    num_local_around_best = 1
    noise_scale_best = 0.2
    for _ in range(num_local_around_best):
        noise = torch.randn(B, A, M, device=device) * noise_scale_best
        mis_loc_best = full_best + noise
        mis_loc_best = torch.clamp(mis_loc_best, 0.0, 1.0)
        mis_list.append(mis_loc_best)

    # ===== Stack all candidate initializations =====
    localMisreports = torch.stack(mis_list, dim=1)           # [B, K, A, M]
    batchMisreports = localMisreports.clone().detach()
    batchMisreports.requires_grad = True


    opt = Adam([batchMisreports], lr=gamma)
    
    for k in range(R):
        advU         = misreportUtility(eval_mechanism,batchTrueValuations,batchMisreports)
        los          =  -1*torch.mean(advU).to(device)
        los.backward()
        opt.step(restricted= True, min=minimum, max=maximum)
        opt.zero_grad()
    
    misReportUtilityMax  = torch.max(advU, dim =1)[0]
    eval_mechanism.zero_grad()
    allocation, payment = eval_mechanism(batchTrueValuations)
    regret = F.relu(misReportUtilityMax -utility(batchTrueValuations, allocation, payment))
    mregret= torch.sum(torch.mean(regret, dim=0)).to(device)
    mregret= float(mregret.cpu().detach().numpy())

    with torch.no_grad():
        l,rMean,p = loss(payment, regret)

    testRegret.append(mregret)
    testPayment.append(float(p.detach().cpu().numpy() ))
    testOptimal.append(float((-l).detach().cpu().numpy())**2)
    
    print("Total regret: ",'{0:.5f}'.format(mregret), "Average regret per bidder: ",'{0:.5f}'.format(mregret/nAgent), " Optimal Revenue: ",'{0:.3f}'.format(float((-l).detach().cpu().numpy())**2), " payment: ",'{0:.3f}'.format(float(p.detach().cpu().numpy() )))

# Initializing Networks

In [5]:
nAgent   = 5
nObject  = 10

# Parameters for the mechanism (payment and allocation network)
nLayersAllocation   = 8
nLayersPayment      = 8
widthAllocation     = 200
widthPayment        = 200

# Parameters for the misreport network
nLayersMisreport    = 8
widthMisreport      = 200

gamma              = 0.001 
testBatch          = 10000

nExperiments       = 200000
batchSize          = 500
nbrBatches         = int(nExperiments/batchSize)


mechanism_base            = AdditiveMechanism(nAgent, nObject, nLayersAllocation, widthAllocation).to(device)
mechanism_base= torch.load("510nips2.pth")
mechanism = MixedWrapper(mechanism_base, reserve=0.5, straight_through=True).to(device)
mechanism.train() 
optimizerMechanism   = AdamW(mechanism_base .parameters(), lr=0.0001)

misreport            = Misreports(nAgent,nObject,nLayersMisreport, widthMisreport).to(device)
optimizerMisreport   = AdamW(misreport.parameters(), lr=0.001)

In [6]:
testRegret    = []
testMaxRegret = []
testPayment   = []
testOptimal   = []
testTime      = []
testIteration = [0]

# range of valuations
minimum            = 0
maximum            = 1

# Training

In [7]:
duration   = 0
R          = 100

i=0

print("Initial Test")
test(50, nbrInit=300, R=300, gamma=0.001, minimum=0, maximum=1)

for t in range(1,60*nbrBatches+1):
    
    # Reinitialize Misreport network periodically at the beginning of training
    if (t%(2*nbrBatches) ==1):
      if   t< 20*nbrBatches+2 :
    
        misreport            = Misreports(nAgent,nObject,nLayersMisreport, widthMisreport).to(device)
        optimizerMisreport   = AdamW(misreport.parameters(), lr=0.001)

    batchTrueValuations = torch.tensor(np.random.rand(batchSize,nAgent,nObject)).float().to(device)
    
    # Optimize Misreport Network for R steps
    for k in range(R):
  
        misreports          = misreport(batchTrueValuations).unsqueeze(1)
        mUtility            = misreportUtility(mechanism,batchTrueValuations,misreports).squeeze(1)
        mLoss               = torch.sum(torch.mean(-mUtility,dim=0))

        optimizerMisreport.zero_grad()
        mLoss.backward()
        optimizerMisreport.step()

    
    # Optimize Mechanism network for one step
    misreports          = misreport(batchTrueValuations).unsqueeze(1)
    mUtility            = misreportUtility(mechanism,batchTrueValuations,misreports).squeeze(1)

    allocation, payment = mechanism(batchTrueValuations)

    regret     = F.relu(mUtility -utility(batchTrueValuations, allocation, payment))
    l,rMean,p = loss(payment, regret)
        
    optimizerMechanism.zero_grad()

    l.backward()

    optimizerMechanism.step()
    
    # Test mechanism periodically
    if t % (2*nbrBatches)==0 :
        print("Batch: ", 2*int(t/(2*nbrBatches)))
        testTime.append(duration)
        testIteration.append(t/nbrBatches)
        test(50, nbrInit=300, R=300, gamma=0.001, minimum=0, maximum=1)

Initial Test


/home/wkw/ysy/hunhe/restrictedAdam.py:103: UserWarning: This overload of add_ is deprecated:
	add_(Number alpha, Tensor other)
Consider using one of the following signatures instead:
	add_(Tensor other, *, Number alpha) (Triggered internally at  ../torch/csrc/utils/python_arg_parser.cpp:1050.)
  exp_avg.mul_(beta1).add_(1 - beta1, grad)


[NET  ] regret=0.01082  avg/bidder=0.00216  optRev=6.258  payment=6.845
[MIXED] regret=0.11438  avg/bidder=0.02288  optRev=4.746  payment=6.923
Batch:  2
[NET  ] regret=0.01765  avg/bidder=0.00353  optRev=6.121  payment=6.888
[MIXED] regret=0.12137  avg/bidder=0.02427  optRev=4.651  payment=6.898
Batch:  4
[NET  ] regret=0.02021  avg/bidder=0.00404  optRev=6.034  payment=6.858
[MIXED] regret=0.14060  avg/bidder=0.02812  optRev=4.443  payment=6.882
Batch:  6
[NET  ] regret=0.01981  avg/bidder=0.00396  optRev=6.037  payment=6.852
[MIXED] regret=0.12156  avg/bidder=0.02431  optRev=4.626  payment=6.870
Batch:  8
[NET  ] regret=0.01918  avg/bidder=0.00384  optRev=6.064  payment=6.865
[MIXED] regret=0.10817  avg/bidder=0.02163  optRev=4.790  payment=6.894
Batch:  10
[NET  ] regret=0.01862  avg/bidder=0.00372  optRev=6.218  payment=7.015
[MIXED] regret=0.11771  avg/bidder=0.02354  optRev=4.798  payment=7.029
Batch:  12
[NET  ] regret=0.01980  avg/bidder=0.00396  optRev=6.046  payment=6.862
[M

# Testing

In [8]:
for i in range(100):
    finaltest(100, nbrInit=300, R=300, gamma=0.001, minimum=0, maximum=1)

Total regret:  0.02218 Average regret per bidder:  0.00444  Optimal Revenue:  6.023  payment:  6.892
Total regret:  0.02399 Average regret per bidder:  0.00480  Optimal Revenue:  6.007  payment:  6.916
Total regret:  0.02326 Average regret per bidder:  0.00465  Optimal Revenue:  5.946  payment:  6.835
Total regret:  0.02270 Average regret per bidder:  0.00454  Optimal Revenue:  6.028  payment:  6.909
Total regret:  0.02304 Average regret per bidder:  0.00461  Optimal Revenue:  6.021  payment:  6.910
Total regret:  0.02344 Average regret per bidder:  0.00469  Optimal Revenue:  6.013  payment:  6.910
Total regret:  0.02391 Average regret per bidder:  0.00478  Optimal Revenue:  5.962  payment:  6.866
Total regret:  0.02328 Average regret per bidder:  0.00466  Optimal Revenue:  5.987  payment:  6.879
Total regret:  0.02280 Average regret per bidder:  0.00456  Optimal Revenue:  6.008  payment:  6.891
Total regret:  0.02371 Average regret per bidder:  0.00474  Optimal Revenue:  5.979  paymen

Total regret:  0.02313 Average regret per bidder:  0.00463  Optimal Revenue:  6.078  payment:  6.972
Total regret:  0.02357 Average regret per bidder:  0.00471  Optimal Revenue:  6.000  payment:  6.899
Total regret:  0.02238 Average regret per bidder:  0.00448  Optimal Revenue:  6.119  payment:  7.000
Total regret:  0.02420 Average regret per bidder:  0.00484  Optimal Revenue:  5.975  payment:  6.886
Total regret:  0.02393 Average regret per bidder:  0.00479  Optimal Revenue:  6.009  payment:  6.917
Total regret:  0.02397 Average regret per bidder:  0.00479  Optimal Revenue:  5.941  payment:  6.845
Total regret:  0.02339 Average regret per bidder:  0.00468  Optimal Revenue:  5.948  payment:  6.839
Total regret:  0.02479 Average regret per bidder:  0.00496  Optimal Revenue:  5.992  payment:  6.918
Total regret:  0.02261 Average regret per bidder:  0.00452  Optimal Revenue:  6.000  payment:  6.877
Total regret:  0.02229 Average regret per bidder:  0.00446  Optimal Revenue:  6.121  paymen

In [9]:
totalregret = np.mean(np.array(testRegret[-100:]))
revenue     = np.mean(np.array(testPayment[-100:]))
print("Final Result")
print("Total Regret = ", '{0:.5f}'.format(totalregret), "Average regret per bidder: ",'{0:.5f}'.format(totalregret/nAgent), " Optimal Revenue: ",'{0:.3f}'.format(float(np.sqrt(revenue)-np.sqrt(totalregret))**2), " payment: ",'{0:.3f}'.format(revenue))

Final Result
Total Regret =  0.02319 Average regret per bidder:  0.00464  Optimal Revenue:  6.117  payment:  6.893


In [10]:
stdregret = np.std(np.array(testRegret[-200:]))
stdrevenue= np.std(np.array(testPayment[-200:]))
print("std Regret = ", '{0:.5f}'.format(stdregret), "std regret per bidder: ",'{0:.5f}'.format(stdregret/nAgent), " std payment: ",'{0:.3f}'.format(stdrevenue))

std Regret =  0.00269 std regret per bidder:  0.00054  std payment:  0.052


In [11]:
torch.save(mechanism, "55stage3.pt")